# 3D Slicer on Google Colab

Run **[3D Slicer](https://www.slicer.org/)** — the medical-imaging application — right inside a Colab
notebook. It runs on the Colab backend and streams into the cell below, with full mouse and keyboard.
Good for slice viewing, segmentation overlays, and light 3D.

Use a **Chrome**-based browser; a plain **CPU runtime** is fine. Run the cells top to bottom — Slicer
appears in the last cell.


## 1. Install dependencies (~1 min)


In [ ]:
%%bash
set -e
cd /content
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq
# Xvfb + Mesa llvmpipe + GStreamer (4 plugin pkgs) + matchbox (single full-screen app WM, no virtual
# desktops) + DejaVu fonts (Slicer's UI font) + Slicer's Qt/QtWebEngine runtime libs.
apt-get install -y -qq --no-install-recommends \
  xvfb xclip matchbox-window-manager fonts-dejavu-core libgl1-mesa-dri libglu1-mesa \
  gstreamer1.0-plugins-base gstreamer1.0-plugins-good gstreamer1.0-plugins-bad gstreamer1.0-plugins-ugly \
  python3-gi gir1.2-gstreamer-1.0 gir1.2-gst-plugins-base-1.0 python3-xlib \
  libxcb-icccm4 libxcb-image0 libxcb-keysyms1 libxcb-randr0 libxcb-render-util0 libxcb-shape0 \
  libxcb-sync1 libxcb-xfixes0 libxcb-xinerama0 libxcb-xkb1 libxkbcommon-x11-0 libxcb-cursor0 libxcb-util1 \
  libodbc2 libpq5 libpulse-mainloop-glib0 libpcre2-16-0 \
  libxcomposite1 libxdamage1 libxtst6 libhwloc15 libnspr4 libnss3 >/dev/null
# libasound2/libcups2 were renamed *t64 in Ubuntu 24.04; Colab is 22.04 -- install whichever exists
apt-get install -y -qq --no-install-recommends libasound2 libcups2 >/dev/null 2>&1 \
  || apt-get install -y -qq --no-install-recommends libasound2t64 libcups2t64 >/dev/null
pip install -q websockets aioquic
echo 'deps installed'

## 2. Get Desktopia + 3D Slicer


In [ ]:
%%bash
set -e
cd /content
REPO=${DESKTOPIA_REPO:-https://github.com/pieper/desktopia}
BRANCH=${DESKTOPIA_BRANCH:-software-render}
rm -rf /content/desktopia
git clone -q --branch "$BRANCH" "$REPO" /content/desktopia || git clone -q "$REPO" /content/desktopia
if ! ls -d /opt/Slicer-*/ >/dev/null 2>&1; then
  echo 'downloading 3D Slicer (~400 MB)...'
  curl -L --retry 3 'https://download.slicer.org/download?os=linux&stability=release' | tar -xz -C /opt
fi
ls -d /opt/Slicer-*/

## 3. Launch Slicer + the stream

`STARTUP` (below) is optional Python that runs **inside Slicer** after it launches — use it to load
data automatically (see the examples under the next cell). `WIDTH/HEIGHT/FPS` size the desktop; the
defaults suit a Colab cell. Re-run this cell to restart.


In [ ]:
import os, time, glob, pathlib, subprocess
os.chdir('/content')
WORK = '/content/desktopia'

WIDTH, HEIGHT, FPS, BITRATE = 1280, 720, 15, 4000
STARTUP = ''   # e.g. 'import SampleData; SampleData.SampleDataLogic().downloadMRHead()'

env = dict(os.environ, DISPLAY=':2', LIBGL_ALWAYS_SOFTWARE='1', GALLIUM_DRIVER='llvmpipe', HOME='/root')

for pat in ('server.py', 'SlicerApp-real'):
    subprocess.run(['pkill', '-f', pat], check=False)
for proc in ('Xvfb', 'matchbox-window-manager'):
    subprocess.run(['pkill', '-x', proc], check=False)
time.sleep(1)

# self-signed cert (the server requires one even though this notebook uses the WebSocket transport)
subprocess.run('openssl req -x509 -newkey ec -pkeyopt ec_paramgen_curve:prime256v1 '
               '-keyout /tmp/k.pem -out /tmp/c.pem -days 1 -nodes -subj /CN=desktopia',
               shell=True, check=True, stderr=subprocess.DEVNULL)

# virtual display
subprocess.Popen(f'Xvfb :2 -screen 0 {WIDTH}x{HEIGHT}x24 +extension GLX +render -noreset',
                 shell=True, env=env, stdout=open('/tmp/xvfb.log','w'), stderr=subprocess.STDOUT)
for _ in range(80):
    if os.path.exists('/tmp/.X11-unix/X2'): break
    time.sleep(0.25)

# matchbox: a single full-screen app, no virtual desktops (so the scroll wheel only ever reaches
# Slicer) -- it auto-maximizes the Slicer window to fill the display.
subprocess.Popen('matchbox-window-manager -use_titlebar no', shell=True, env=env,
                 stdout=open('/tmp/wm.log','w'), stderr=subprocess.STDOUT)
time.sleep(1)

# 3D Slicer, optionally running your STARTUP script once it's up
SDIR = sorted(glob.glob('/opt/Slicer-*/'))[0]
cmd = f'{SDIR}/Slicer --no-splash'
if STARTUP.strip():
    pathlib.Path('/tmp/slicer_startup.py').write_text(STARTUP)
    cmd += ' --python-script /tmp/slicer_startup.py'
subprocess.Popen(cmd, shell=True, env=env, stdout=open('/tmp/slicer.log','w'), stderr=subprocess.STDOUT)

# tell the page to use the WebSocket transport, then start the server (page + WS on one port)
pathlib.Path(f'{WORK}/client/status.json').write_text('{"ready":true,"transport":"websocket"}')
subprocess.Popen('python3 server.py --cert /tmp/c.pem --key /tmp/k.pem '
                 f'--source xvfb --width {WIDTH} --height {HEIGHT} --fps {FPS} --bitrate {BITRATE} '
                 '--ws-plain --serve-dir client',
                 shell=True, env=env, cwd=WORK,
                 stdout=open('/tmp/server.log','w'), stderr=subprocess.STDOUT)
time.sleep(8)
print('--- server.log ---'); print(open('/tmp/server.log').read()[-1500:])
print('Slicer is still loading; the view appears in the cell below shortly.')

### Loading data at startup

Set `STARTUP` in the cell above to Python that runs inside Slicer. A few examples:

Built-in sample volume:
```python
STARTUP = 'import SampleData; SampleData.SampleDataLogic().downloadMRHead()'
```

Load a NIfTI/NRRD from a URL:
```python
STARTUP = 'slicer.util.loadVolume("https://example.org/scan.nrrd")'
```

[Imaging Data Commons (IDC)](https://imaging.datacommons.cancer.gov/) — pull a series by UID:
```python
STARTUP = '''
slicer.util.pip_install("idc-index")
from idc_index import IDCClient
IDCClient().download_from_selection(seriesInstanceUID="<SERIES_UID>", downloadDir="/root/Data")
from DICOMLib import DICOMUtils
with DICOMUtils.TemporaryDICOMDatabase() as db:
    DICOMUtils.importDicom("/root/Data", db)
    DICOMUtils.loadPatientByUID(list(db.patients())[0])
'''
```


## 4. Open Slicer in this cell

Click into the frame to focus it, then use the mouse and keyboard normally.


In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(4434, path='/index.html', height=600, cache_in_notebook=False)

### Troubleshooting

- **Check Slicer's log:** `print(open('/tmp/slicer.log').read())`
- **Frame stuck on "waiting for launcher…" / "Connecting":** open it in a tab instead:
  ```python
  from google.colab import output
  output.serve_kernel_port_as_window(4434, path='/index.html')
  ```
- **Black 3D / no slices:** `!apt-get install -y mesa-utils && glxinfo -B` should say *llvmpipe*.
- **Stream never starts:** check `/tmp/server.log` for a GStreamer error.
